# 🦅 PREDATOR Analytics v67.0-ELITE — Kaggle Production Node

**Режим**: CPU Only, Max RAM 30GB  
**Тунель**: zrok (OpenZiti)  
**БД**: 10 DB Architecture (SQLite + in-memory mocks)  

## ⚙️ Перед запуском:
1. **Internet ON** (Add-ons → Internet)
2. **Secrets**: додати `ZROK_TOKEN` (Add-ons → Secrets)
3. Натиснути **Run All** (⏩)

---

In [ ]:
# ─── Cell 1: Налаштування Secrets та середовища ────────────────
import os
import sys

print('🔐 Завантаження Kaggle Secrets...')
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_SECRET_ZROK_TOKEN'] = secrets.get_secret('ZROK_TOKEN')
    print(f'✅ ZROK_TOKEN завантажено: {os.environ["KAGGLE_SECRET_ZROK_TOKEN"][:4]}****')
except Exception as e:
    print(f'⚠️ Kaggle Secrets недоступні: {e}')
    # Fallback — встановити вручну нижче:
    os.environ['KAGGLE_SECRET_ZROK_TOKEN'] = ''  # <-- ВСТАВТЕ ВАШ ZROK TOKEN
    print('ℹ️  Встановіть ZROK_TOKEN вручну у рядку вище')

# Конфігурація
os.environ['PREDATOR_DB_DIR'] = '/kaggle/working'
os.environ['PREDATOR_SECRET_KEY'] = 'predator-kaggle-prod-key-v67-change-in-prod'

print(f'📁 DB Directory: {os.environ["PREDATOR_DB_DIR"]}')
print(f'🐍 Python: {sys.version}')

In [ ]:
# ─── Cell 2: Встановлення залежностей ─────────────────────────
import subprocess

print('📦 Встановлення залежностей...')
deps = [
    'fastapi',
    'uvicorn[standard]',
    'psutil',
    'httpx',
    'python-jose[cryptography]',
    'sqlalchemy',
    'aiosqlite',
    'networkx',
    'orjson',
    'numpy',
    'sse-starlette',
    'nest_asyncio',
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *deps],
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ Всі залежності встановлено')
else:
    print(f'⚠️ Помилка: {result.stderr[-500:]}')

# Перевірка
import fastapi, uvicorn, psutil, networkx, numpy  # noqa
print(f'   fastapi={fastapi.__version__}')
print(f'   uvicorn={uvicorn.__version__}')
print(f'   networkx={networkx.__version__}')
print(f'   numpy={numpy.__version__}')

In [ ]:
# ─── Cell 3: Завантаження бекенду ─────────────────────────────
import urllib.request

BACKEND_PATH = '/kaggle/working/predator_backend.py'

# Варіант 1: З GitHub (якщо репозиторій публічний)
GITHUB_URL = 'https://raw.githubusercontent.com/dima1203oleg/predator-analytics/main/scripts/predator_kaggle_prod_v67.py'

try:
    print('🔽 Завантаження бекенду з GitHub...')
    req = urllib.request.Request(GITHUB_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=30) as resp:
        content = resp.read()
    with open(BACKEND_PATH, 'wb') as f:
        f.write(content)
    print(f'✅ Бекенд завантажено: {len(content):,} bytes')
except Exception as e:
    print(f'⚠️ GitHub недоступний: {e}')
    print('ℹ️  Вставте код бекенду напряму у наступну клітинку або завантажте файл вручну')
    # Fallback — записати мінімальний backend
    minimal = '''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
app = FastAPI(title="PREDATOR Minimal")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
@app.get("/api/v1/health")
async def health(): return {"status": "ONLINE", "mode": "MINIMAL_FALLBACK"}
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''
    with open(BACKEND_PATH, 'w') as f:
        f.write(minimal)
    print('⚠️ Записано мінімальний fallback backend')

print(f'📁 Шлях: {BACKEND_PATH}')

In [ ]:
# ─── Cell 4: Запуск PREDATOR Backend ──────────────────────────
# ⚠️ Ця клітинка блокуюча — сервер працює до зупинки
# 🔗 PUBLIC URL з'явиться у виводі нижче

print('🦅 Запуск PREDATOR Analytics v67.0-ELITE...')
print('⏳ Зачекайте ~60 секунд для повної ініціалізації')
print('-' * 60)

import nest_asyncio

nest_asyncio.apply()

# Виконуємо бекенд
exec(open(BACKEND_PATH).read())

---
## 📋 Після запуску

1. Скопіювати `PUBLIC URL` з виводу вище
2. Відкрити файл `.env.local` у frontend:
   ```
   /Users/Shared/Predator_60/apps/predator-analytics-ui/.env.local
   ```
3. Замінити `VITE_API_URL`:
   ```env
   VITE_API_URL=https://YOUR-URL.share.zrok.io/api/v1
   ```
4. Перезапустити frontend: `npm run dev`

### 🔑 Логін
| Логін | Пароль | Роль |
|-------|--------|------|
| `admin` | `admin123` | Адміністратор |
| `analyst` | `analyst123` | Аналітик |
| `operator` | `operator123` | Оператор |
| `viewer` | `viewer123` | Переглядач |